# AI and NLP for Spatial Humanities
## 01 - Manual annotation: humans define the task before models are scored

**Duration:** 35-45 minutes

This notebook begins without an NLP model. We first make the human decisions visible: what counts as spatial evidence, where its boundaries lie, which relations are supported, and what requires inference.

**Learning outcomes:** distinguish location/locale/sense of place; create exact-offset annotations; compare exact and overlap matching; identify selection, boundary, ontology and inference disagreements; save a human annotation for later model comparison.

> A human reference annotation is a documented scholarly decision, not interpretation-free truth.

## 1. Setup

This notebook runs after Notebook 00 or independently. No GPU or API key is required. `FAST_MODE=True` remains the workshop default.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
FAST_MODE = True
cwd = Path.cwd()
if (cwd/"spatio_textual").exists() and (cwd/"projects"/"sh2026").exists():
    repo_dir = cwd
else:
    repo_dir = Path("/content/spatio-textual")
    if not repo_dir.exists():
        subprocess.run(["git","clone","--depth","1","--branch","spatial-humanities-2026",
                        "https://github.com/IgnatiusEzeani/spatio-textual.git",str(repo_dir)],check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements-lite.txt"],check=True)
print("Repository:", repo_dir)

In [ ]:
import pandas as pd
from IPython.display import display
from spatio_textual.gold import (
    SPAN_LABELS, assert_valid_gold, find_span, load_gold_jsonl,
    score_span_annotations, select_spans,
)
gold_path = repo_dir/"projects"/"sh2026"/"workshop"/"data"/"gold_reference_v0.1.jsonl"
records = load_gold_jsonl(gold_path)
assert_valid_gold(records)
gold = {r["example_id"]: r for r in records}
print(f"Loaded {len(records)} validated reference examples.")

## 2. What counts as spatial information?

We use three connected analytical levels:

| Level | Question | Examples |
|---|---|---|
| **Location** | What named place is mentioned? | Penrith, Eamont |
| **Locale** | What spatial setting/feature is described? | road, river, village, woods |
| **Sense of place** | How is place experienced/evaluated? | picturesque, wild, sensory/affective description |

Spatial narratives also encode distance, direction, time, movement, relations and uncertainty. They cannot be reduced to coordinates.

In [ ]:
record = gold["cldw_penrith_pooley_bridge"]
text = record["text"]
print(text)
print("\nReference status:", record["reference_status"])
print("Distribution:", record["source"]["distribution_status"])

## 3. Exercise A - annotate freely

Before reading the label guide, mark anything you regard as spatially meaningful.

Ask yourself:
- Which strings are named places?
- Is **roads** spatial information?
- Is **about six miles distant** spatial information?
- Does **spans the Eamont** encode something that a place-name recognizer would miss?
- What would be lost if we retained only latitude/longitude?

The ontology decides what a system is capable of seeing **before any model is run**.

### Working labels
`TOPONYM`, `GEONOUN`, `SPATIAL_RELATION`, `DISTANCE`, `DIRECTION`, `TIME`, `MOVEMENT_CUE`, `TRANSPORT_CUE`, `SUBJECTIVE_DESCRIPTOR`, `SENSORY_DESCRIPTOR`, `DEICTIC_REFERENCE`.

See `projects/sh2026/docs/GOLD_ANNOTATION_GUIDE.md` for the full policy.

In [ ]:
guide = pd.DataFrame([
("TOPONYM","entity","location"),("GEONOUN","entity","locale"),
("SPATIAL_RELATION","spatial_cue",None),("DISTANCE","spatial_cue",None),
("DIRECTION","spatial_cue",None),("TIME","temporal_cue",None),
("MOVEMENT_CUE","event_cue",None),("TRANSPORT_CUE","journey_cue",None),
("SUBJECTIVE_DESCRIPTOR","sense_of_place","sense_of_place"),
("SENSORY_DESCRIPTOR","sense_of_place","sense_of_place"),
("DEICTIC_REFERENCE","spatial_cue","location")],
columns=["label","layer","conceptual_level"])
display(guide)

## 4. Exact offsets are part of the evidence

Offsets are half-open: `text[start_char:end_char]` must reproduce the source string exactly. The helper below refuses to invent an occurrence that is not present.

In [ ]:
example = find_span(text,"Penrith","TOPONYM",layer="entity",span_id="p001")
assert text[example["start_char"]:example["end_char"]] == example["text"]
example

## 5. Exercise B - build your annotation

Edit the list. The starter contains only one span so the instructor reference is not handed to you.

In [ ]:
participant_spans = [
    find_span(text,"Penrith","TOPONYM",layer="entity",span_id="p001"),
    # Add your own decisions, for example:
    # find_span(text,"roads","GEONOUN",layer="entity",span_id="p002"),
]
for s in participant_spans:
    assert s["label"] in SPAN_LABELS
    assert text[s["start_char"]:s["end_char"]] == s["text"]
display(pd.DataFrame(participant_spans))

## 6. Predict the disagreements before revealing the reference

Discuss:
1. **selection** - did you include `roads`?
2. **boundary** - `six miles`, `about six miles`, or `about six miles distant`?
3. **ontology** - is a road locale, infrastructure, both, or outside the task?
4. **anaphora/inference** - does resolving *which* in `which spans the Eamont` require interpretation?
5. **historical form** - should `Ulleswater` be silently normalized?
6. **representation** - should route structure be a span, a relation, or a journey?

These disagreements are methodological evidence, not mere noise.

In [ ]:
SHOW_REFERENCE = True  # set only after discussion
reference_spans = record["spans"]
if SHOW_REFERENCE:
    display(pd.DataFrame(reference_spans)[
        ["span_id","text","label","layer","conceptual_level","start_char","end_char",
         "certainty","attributes","notes"]
    ])

## 7. Score the spans, then read the disagreements

We report:
- **exact**: same label and exact boundaries;
- **overlap**: same label and overlapping evidence, one-to-one matched by maximum IoU.

Neither score replaces qualitative inspection.

In [ ]:
scores=[]
for mode in ("exact","overlap"):
    result=score_span_annotations(participant_spans,reference_spans,match=mode,label_sensitive=True)
    scores.append({k:v for k,v in result.items() if k in {"match","precision","recall","f1","tp","fp","fn"}})
display(pd.DataFrame(scores))

exact=score_span_annotations(participant_spans,reference_spans,match="exact")
print("Unmatched participant rows:")
display(pd.DataFrame([participant_spans[i] for i in exact["unmatched_pred_indices"]]))
print("Unmatched reference rows:")
display(pd.DataFrame([reference_spans[i] for i in exact["unmatched_ref_indices"]]))

## 8. A reasonable boundary choice can still get exact F1 = 0

The reference marks `about six miles distant`. A second annotator could reasonably mark only `six miles`. Exact matching measures boundary agreement; overlap matching measures shared evidence.

In [ ]:
distance_ref = select_spans(record,["DISTANCE"])
annotator_b = [find_span(text,"six miles","DISTANCE",layer="spatial_cue")]
display(pd.DataFrame([
    {"match":"exact", **{k:v for k,v in score_span_annotations(annotator_b,distance_ref,match="exact").items() if k in {"precision","recall","f1"}}},
    {"match":"overlap", **{k:v for k,v in score_span_annotations(annotator_b,distance_ref,match="overlap").items() if k in {"precision","recall","f1"}}},
]))

## 9. Spans are not enough: inspect relations

A relation record separates the wording from the interpreted relation and records whether the interpretation is explicit or contextual.

In [ ]:
display(pd.DataFrame(record["relations"])[
    ["relation_id","type","source_span_id","target_span_id",
     "evidence_quote","certainty","attributes","notes"]
])

Notice that a model can recognize every toponym and still miss route connectivity, approximate distance, anaphoric `SPANS`, or the relation expressed by `issue from`.

This is why later benchmarks separate:
- entity/span extraction;
- entity resolution;
- relation extraction;
- journey reconstruction.

## 10. Save the participant annotation

We preserve the human decision rather than overwriting it with the instructor reference.

In [ ]:
out = repo_dir/"sh2026_outputs"/"human_review"
out.mkdir(parents=True,exist_ok=True)
participant_record = {
    "schema_version":"sh2026-participant-0.1",
    "example_id":record["example_id"],
    "text":text,
    "reference_schema":record["schema_version"],
    "annotation_policy":record["annotation_policy"],
    "spans":participant_spans,
    "relations":[],
    "notes":["Created independently of later computational model comparison."],
}
path=out/f"{record['example_id']}_participant.json"
path.write_text(json.dumps(participant_record,indent=2,ensure_ascii=False),encoding="utf-8")
print("Saved:",path)

## 11. Extension - spatial meaning without conventional toponyms

The synthetic passage below is deliberately rich in locale, relative position and vague spatial language. It should not be forced into a set of coordinates.

In [ ]:
rel = gold["synthetic_relational_space"]
print(rel["text"])
print("\nQuestions: Which items are locale? Is `nearest` a distance? Is `to our left` a direction without a compass frame? Can village → woods be a journey? Is 'hide' an explicit reason or an inference?")
display(pd.DataFrame(rel["spans"])[["text","label","layer","certainty","attributes"]])
display(pd.DataFrame(rel["relations"])[["type","evidence_quote","certainty","attributes","notes"]])
display(pd.DataFrame(rel["journeys"])[["start_location","end_location","date","journey_reason",
                                      "explicit_or_inferred","requires_review","review_notes"]])

## 12. Data governance and takeaways

The oral-history-style examples in the public tutorial are **synthetic** and labelled as such. Controlled-access Holocaust testimony text is not bundled into this public reference set.

Before any model is evaluated, humans have already decided:
1. what the task is;
2. which concepts matter;
3. where boundaries lie;
4. which relations warrant annotation;
5. how much inference is acceptable;
6. how historical/ambiguous geography is treated;
7. what evidence must be preserved.

So our later comparison will ask not only **which method has the highest F1?**, but also:

> **Which method makes its assumptions, uncertainty and evidential basis easiest for a humanities researcher to inspect and correct?**

Next: **02 - Rules and gazetteers**.